### Libraries

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotnine import ggplot, aes, geom_point, scale_x_log10, scale_y_continuous, scale_color_manual, labs, theme_bw, theme, element_text
import matplotlib.gridspec as gridspec
import os
import pickle

group = "Panthera_leo"
group = "Ceratotherium_simum"
#group = "Rhinoceros_unicornis"
#group = "Boselaphus_tragocamelus"
ref = "Tragelaphus_eurycerus_isaaci"


if os.getcwd().startswith("/home/lakrids"):
    path_prefix = "/home/lakrids/GenomeDK"
else:
    path_prefix = "/faststorage/project/"
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/population_list.txt") as f:
        pops = [line.strip() for line in f]
population_name = pops[0]
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
    parameters = pickle.load(file)

if group == "Boselaphus_tragocamelus":
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref}/ref/regions_{ref}_updated.txt", delimiter="\t")
else: 
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/ref/regions_{group}_updated.txt", delimiter="\t")

    
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}_filtered.txt") as f:
    samples = [line.strip() for line in f if line.strip()]


### File Input & First Data Impression

In [10]:
df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

chromosomes = df_regions[
        (df_regions["female_ploidy"] == 2) &
        (df_regions["male_ploidy"] == 2) &
        (df_regions["end"] > 20000000)
    ]["region"].tolist()

# Build a contig length dict from the regions df
contig_lengths = df_regions.set_index("region")["end"].to_dict()

df_stats = df_stats[["sample"] + chromosomes]
# Divide each chromosome column by its contig lengths
for chrom in df_stats.columns:
    if chrom in contig_lengths:
        df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
df_stats

,sample,NW_004454156.1,NW_004454157.1,NW_004454158.1,NW_004454159.1,NW_004454160.1,NW_004454161.1,NW_004454162.1,NW_004454163.1,NW_004454164.1,...,NW_004454189.1,NW_004454190.1,NW_004454191.1,NW_004454192.1,NW_004454193.1,NW_004454194.1,NW_004454195.1,NW_004454196.1,NW_004454197.1,NW_004454198.1
0,SAMEA8896052,0.054868,0.055300,0.049275,0.059150,0.055257,0.040978,0.064073,0.043889,0.051716,...,0.047206,0.068494,0.056444,0.037703,0.050571,0.052692,0.047955,0.036915,0.074985,0.042230
1,SAMEA8896053,0.063512,0.066592,0.057165,0.069518,0.065566,0.061632,0.073777,0.059719,0.071176,...,0.052662,0.079357,0.078818,0.055195,0.068786,0.056165,0.064006,0.055615,0.084171,0.072242
2,SAMEA8896055,0.050024,0.049781,0.045884,0.054413,0.050487,0.044793,0.057812,0.046001,0.054664,...,0.043537,0.062675,0.058813,0.039767,0.047686,0.049062,0.050858,0.033681,0.068095,0.042002
3,SAMEA8896056,0.056472,0.067690,0.051720,0.059959,0.056081,0.050799,0.063156,0.052540,0.060166,...,0.050056,0.084450,0.071367,0.046626,0.054906,0.055002,0.056863,0.040312,0.081889,0.048686
4,SAMEA8896057,0.050699,0.049635,0.046697,0.054580,0.049740,0.045547,0.057380,0.047522,0.054454,...,0.044532,0.072086,0.058335,0.041112,0.049409,0.050714,0.050813,0.036537,0.071441,0.044903
5,SAMEA8896058,0.047810,0.056019,0.043021,0.059038,0.055256,0.042980,0.062985,0.044623,0.051676,...,0.041449,0.068270,0.064808,0.038254,0.045760,0.046946,0.048055,0.032725,0.066915,0.048405
6,SAMN21971314,0.048393,0.046612,0.043526,0.051706,0.047285,0.043643,0.054165,0.043757,0.052170,...,0.041522,0.064985,0.054844,0.037306,0.048618,0.047865,0.047406,0.035875,0.062450,0.050261
7,SAMN21971315,0.049897,0.049580,0.044750,0.054360,0.049916,0.045902,0.056937,0.046805,0.054787,...,0.042625,0.068846,0.059103,0.040572,0.048753,0.049791,0.049658,0.037698,0.066323,0.048301
8,mappability,0.008985,0.007765,0.008478,0.008632,0.009898,0.009184,0.008125,0.007189,0.009455,...,0.006253,0.008099,0.010871,0.009651,0.011240,0.007201,0.008425,0.006840,0.010376,0.008569
9,coverage,0.151376,0.161598,0.140601,0.154188,0.165465,0.143904,0.158636,0.127294,0.150811,...,0.149122,0.209316,0.160345,0.119144,0.143252,0.138291,0.125361,0.158233,0.222434,0.167869


In [ ]:
df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

In [7]:
genus_list      = ["Loxodonta", "Elephas", "Boselaphus", "Panthera", "Rhinoceros", "Ceratotherium", "Diceros"]
#genus_list = ["Elephas"]
population_name = "pop"

data            = pd.concat([pd.read_table(f) for f in [f"{path_prefix}/megaFauna/sa_megafauna/metadata/samples_{genus}.txt" for genus in genus_list]], ignore_index=True) # add all SRR accession from genus_list in a single data frame
data            = data.reset_index(drop=True)
ref_folders     = sorted(set(data.REFERENCE_FOLDER)) # list of references needed to map the SRR accessions

references      = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/metadata/references.txt")
references      = references.loc[references.REFERENCE_FOLDER.isin(ref_folders)] # filter references to keep only the ones needed to map the SRR accessions
references      = references.reset_index(drop=True)

## make data frame that contains names of species-specific folders and the reference folders used to map the species
species_and_refs = pd.DataFrame({"FOLDER": data.FOLDER, "REFERENCE_FOLDER": data.REFERENCE_FOLDER, "GVCF_FOLDER": [data.GENUS.iloc[jj] + "_" + data.SPECIES.iloc[jj] for jj in range(data.shape[0])]}).drop_duplicates()
species_and_refs = species_and_refs.reset_index(drop=True)

## merge dataframes to have species-specific folder, reference folder and fastq ftps in same dataframe
species_and_refs = species_and_refs.merge(references, how = "left")

##########################################
#    	     --- Preparations ---
##########################################

for i in range(species_and_refs.shape[0]):
    # Initialising folders and variables for putting in the functions
    group      = species_and_refs.FOLDER[i]
    ref_folder = species_and_refs.REFERENCE_FOLDER[i]
    
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
        parameters = pickle.load(file)    

    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref_folder}/ref/regions_{ref_folder}_updated.txt", delimiter="\t")
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}.txt") as f:
        samples = [line.strip() for line in f if line.strip()]

    df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

    chromosomes = df_regions[
            df_regions["region"].str.startswith("NC") &
            (df_regions["female_ploidy"] == 2) &
            (df_regions["male_ploidy"] == 2)
        ]["region"].tolist()

    # Build a contig length dict from the regions df
    contig_lengths = df_regions.set_index("region")["end"].to_dict()

    df_stats = df_stats[["sample"] + chromosomes]
    # Divide each chromosome column by its contig lengths
    for chrom in df_stats.columns:
        if chrom in contig_lengths:
            df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
    df_stats

    df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/mask_stats_{group}.txt")

KeyError: "None of [Index(['sample', 'NC_087342.1', 'NC_087343.1', 'NC_087344.1', 'NC_087345.1',\n       'NC_087346.1', 'NC_087347.1', 'NC_087348.1', 'NC_087349.1',\n       'NC_087350.1', 'NC_087351.1', 'NC_087352.1', 'NC_087353.1',\n       'NC_087354.1', 'NC_087355.1', 'NC_087356.1', 'NC_087357.1',\n       'NC_087358.1', 'NC_087359.1', 'NC_087360.1', 'NC_087361.1',\n       'NC_087362.1', 'NC_087363.1', 'NC_087364.1', 'NC_087365.1',\n       'NC_087366.1', 'NC_087367.1', 'NC_087368.1'],\n      dtype='object')] are in the [columns]"